# 11 — Lamin / KG schema explorer

This notebook is meant as a **guided presentation** of the current Jouvence KG surface, not just a low-level debug notebook.

What it shows:

1. **What is configured in LaminDB right now** — which schema modules/imports are available in the active `jkobject/jouvencekb` instance, and which are missing.
2. **What the project KG schema says** — node types, relation contracts, expected edge/evidence files, and Parquet column contracts from `manage_db.kg_schema` / `manage_db.kg_evidence`.
3. **What exists in the actual KG files** — file presence, row counts, columns, and basic schema metadata under a mounted or cached KG root.
4. **What is now possible** — fast read-only inspection of canonical KG structure through the working macOS GCS FUSE mount.
5. **What is still missing / pending** — deferred Lamin schema configuration, deferred relation promotions, and ReMap-dependent gates.

The notebook is intentionally **read-only**. It does not write to Lamin, does not promote KG artifacts, and does not download remote datasets. Think of it as a dashboard/explainer for where the KG currently stands after the recent build/promote/review wave.


## Safety contract and how to read this notebook

Defaults are safe for interactive use:

- `READ_ONLY = True`; this notebook never calls project writer/sync/download functions.
- KG roots are discovered from local/mounted paths only. The current preferred Mac path is:
  - `~/mnt/gcs/jouvencekb-kg/v2` → maps to `gs://jouvencekb/kg/v2` through `gcsfuse --only-dir kg`.
- Lamin diagnostics are **best-effort**. If a schema module is not configured in Lamin, the notebook records that as a table row rather than treating it as a runtime failure.
- Parquet inspection uses metadata/schema reads; expensive full scans are off unless you explicitly opt in via environment flags.

Useful env flags before launching Jupyter:

```bash
export JOUVENCE_KG_ROOT="$HOME/mnt/gcs/jouvencekb-kg/v2"
export JOUVENCE_SCHEMA_FULL_SCAN=0
export JOUVENCE_LAMIN_INTROSPECT=0
```

Interpretation rule: if Lamin says a module is unavailable, that only means the **Lamin registry layer** is not configured. It does not imply that the canonical KG Parquets are missing or invalid.


In [1]:
from __future__ import annotations

import importlib
import inspect
import os
from pathlib import Path
from typing import Any

import pandas as pd
import pyarrow.parquet as pq

try:
    from IPython.display import display
except Exception:  # noqa: BLE001 - keep plain Python execution usable if IPython is unavailable/incompatible
    def display(obj):
        print(obj)

# This notebook is intentionally a reader/viewer. Keep this true unless you are
# deliberately turning it into a writer notebook in a separate reviewed task.
READ_ONLY = True
FULL_SCAN = os.environ.get('JOUVENCE_SCHEMA_FULL_SCAN', os.environ.get('TXGNN_SCHEMA_FULL_SCAN', '0')) == '1'
LAMIN_INTROSPECT = os.environ.get('JOUVENCE_LAMIN_INTROSPECT', os.environ.get('TXGNN_LAMIN_INTROSPECT', '0')) == '1'
REPO_ROOT = Path.cwd()
# Candidate KG roots are ordered from most local/project-specific to older fallback paths.
# The verified macOS mount is ~/mnt/gcs/jouvencekb-kg/v2, which exposes
# gs://jouvencekb/kg/v2 through gcsfuse --only-dir kg.
DEFAULT_KG_ROOT_CANDIDATES = [
    os.environ.get('JOUVENCE_KG_ROOT', os.environ.get('TXGNN_KG_ROOT')),
    '~/mnt/gcs/jouvencekb-kg/v2',
    '/Users/jkobject/mnt/gcs/jouvencekb-kg/v2',
    '/mnt/gcs/jouvencekb/kg/v2',
    # Historical/non-default provenance only: old legacy .omoc local cache, kept last for read-only archaeology and never a default.
    'historical legacy .omoc/gcs-cache/kg-v2 (non-default; provenance-only)',
]
KG_ROOTS = [Path(p).expanduser() for p in DEFAULT_KG_ROOT_CANDIDATES if p]
print('read_only:', READ_ONLY)
print('full_scan:', FULL_SCAN)
print('lamin_introspect:', LAMIN_INTROSPECT)
print('repo_root:', REPO_ROOT)
print('kg_root_candidates:')
for root in KG_ROOTS:
    print('-', root, 'exists=' + str(root.exists()))

read_only: True
full_scan: False
lamin_introspect: False
repo_root: /Users/jkobject/Documents/jouvence/notebooks
kg_root_candidates:
- /Users/jkobject/mnt/gcs/jouvencekb-kg/v2 exists=True
- /Users/jkobject/mnt/gcs/jouvencekb-kg/v2 exists=True
- /mnt/gcs/jouvencekb/kg/v2 exists=False
- historical legacy .omoc/gcs-cache/kg-v2 (non-default; provenance-only) exists=False


## LaminDB / `lnschema` availability diagnostics

This section answers: **what is Lamin configured to know about?**

Current observed state from `uv run lamin info`:

```text
Instance: jkobject/jouvencekb
modules: bionty, pertdb
```

So `lnschema_txgnn` is **not configured in the Lamin instance**. The package/schema code can exist in the repository, but Lamin still refuses to activate it unless the instance settings include that schema module. That is why imports such as `manage_db.lnschema_txgnn.models` can show:

```text
ModuleWasntConfigured: 'lnschema_txgnn' wasn't configured for this instance
```

For this notebook, that is a useful diagnostic, not a blocker. The canonical KG schema below comes from the local project code and Parquet files; the custom Lamin registry layer is a separate future activation/config step.


In [ ]:
# Import each module defensively. Lamin can raise ModuleWasntConfigured for
# schema modules that exist in code but are not enabled for the active instance.
def module_diagnostic(module_name: str) -> dict[str, Any]:
    row: dict[str, Any] = {'module': module_name, 'available': False, 'version': None, 'path': None, 'error': None}
    try:
        module = importlib.import_module(module_name)
        row['available'] = True
        row['version'] = getattr(module, '__version__', None)
        row['path'] = getattr(module, '__file__', None)
    except BaseException as exc:  # noqa: BLE001 - diagnostics should not abort exploration; Lamin can raise ModuleWasntConfigured
        row['error'] = f'{type(exc).__name__}: {exc}'
    return row

LAMIN_MODULES = [
    'lamindb',
    'lnschema_core',
    'lnschema_bionty',
    'bionty',
    'pertdb',
    'manage_db.lnschema_txgnn',
    'manage_db.lnschema_txgnn.models',
]
lamin_module_status = pd.DataFrame([module_diagnostic(name) for name in LAMIN_MODULES])
display(lamin_module_status)

print('Interpretation: available=True means import/config works in this environment; error rows are expected for optional or not-yet-configured schema modules.')

In [ ]:
# If the custom TxGNN Lamin schema becomes configured later, this table will
# enumerate the model classes it exposes. Today, it usually shows the
# ModuleWasntConfigured diagnostic instead.
def class_table(module_name: str) -> pd.DataFrame:
    try:
        module = importlib.import_module(module_name)
    except BaseException as exc:  # noqa: BLE001 - Lamin schema configuration errors are diagnostics here
        return pd.DataFrame([{'module': module_name, 'error': f'{type(exc).__name__}: {exc}'}])
    rows = []
    for name, obj in inspect.getmembers(module, inspect.isclass):
        if getattr(obj, '__module__', '') == module.__name__:
            rows.append({
                'class': name,
                'module': obj.__module__,
                'bases': ', '.join(base.__name__ for base in obj.__bases__),
                'doc': (inspect.getdoc(obj) or '').splitlines()[0] if inspect.getdoc(obj) else '',
            })
    return pd.DataFrame(rows).sort_values(['module', 'class']) if rows else pd.DataFrame(columns=['class', 'module', 'bases', 'doc'])

custom_lamin_classes = class_table('manage_db.lnschema_txgnn.models')
display(custom_lamin_classes)

## Canonical schema from `manage_db.kg_schema`

This is the most important schema source for the current KG build.

It does **not** depend on LaminDB being configured with `lnschema_txgnn`. Instead, it reads the project's explicit KG contract:

- `NODE_TYPES`: canonical node types, ID conventions, xrefs, and expected node Parquet files.
- `RELATIONS`: canonical edge relation names, endpoint types, status, replacements, and notes.
- `EDGE_PARQUET_COLUMNS` / `EVIDENCE_PARQUET_COLUMNS`: baseline file contracts for graph assertions and evidence rows.

Use these tables to understand what the pipeline now expects to exist in `kg/v2/{nodes,edges,evidence}`.


In [ ]:
from manage_db.kg_schema import EDGE_PARQUET_COLUMNS, NODE_TYPES, RELATIONS
from manage_db.kg_evidence import EVIDENCE_PARQUET_COLUMNS


def enum_value(value: Any) -> str:
    return getattr(value, 'value', str(value))

# Build a human-readable table from the canonical node schema.
node_schema_rows = []
for node_type, info in NODE_TYPES.items():
    node_schema_rows.append({
        'node_type': enum_value(node_type),
        'primary_ontology': info.primary_ontology,
        'id_format': info.id_format,
        'bionty_registry': info.bionty_registry,
        'example_id': info.example_id,
        'xref_columns': ', '.join(info.xref_columns),
        'expected_file': f'nodes/{enum_value(node_type)}.parquet',
    })
node_schema = pd.DataFrame(node_schema_rows)
display(node_schema)

In [ ]:
# Build a human-readable table from the canonical relation schema.
relation_schema_rows = []
for relation in RELATIONS:
    relation_schema_rows.append({
        'relation': relation.name,
        'x_type': enum_value(relation.source),
        'y_type': enum_value(relation.target),
        'kind': enum_value(relation.kind),
        'direct': relation.direct,
        'status': enum_value(relation.status),
        'replacement': relation.replacement,
        'edge_file': f'edges/{relation.name}.parquet',
        'evidence_file': f'evidence/{relation.name}.parquet',
        'notes': relation.notes,
    })
relation_schema = pd.DataFrame(relation_schema_rows)
relation_schema

In [ ]:
# These are the common Parquet column contracts every edge/evidence file is checked against.
edge_contract = pd.DataFrame(EDGE_PARQUET_COLUMNS, columns=['column', 'description'])
evidence_contract = pd.DataFrame(EVIDENCE_PARQUET_COLUMNS, columns=['column', 'description'])
print('Canonical edge parquet columns')
display(edge_contract)
print('Canonical evidence parquet columns')
display(evidence_contract)

## Local KG root selection and file inventory

This section answers: **what canonical KG files are actually visible from this environment?**

The preferred path now is the working macOS FUSE mount:

```text
~/mnt/gcs/jouvencekb-kg/v2
```

which maps to:

```text
gs://jouvencekb/kg/v2
```

Use `JOUVENCE_KG_ROOT` (legacy fallback: `TXGNN_KG_ROOT`) or the verified FUSE mount (`/Users/jkobject/mnt/gcs/jouvencekb-kg/v2`) for current reads. The legacy `.omoc/gcs-cache/kg-v2` path is historical/non-default and appears only as the last read-only archaeology fallback. The inventory tables below do metadata reads: existence, row counts from Parquet metadata, column names/types, and file sizes.


In [ ]:
# Pick the first existing KG root. Override with JOUVENCE_KG_ROOT (legacy fallback: TXGNN_KG_ROOT) if you want
# to force the mounted canonical KG rather than a local cache.
def select_existing_root(candidates: list[Path]) -> Path | None:
    for root in candidates:
        if root.exists():
            return root
    return None

KG_ROOT = select_existing_root(KG_ROOTS)
print('selected_kg_root:', KG_ROOT)

In [ ]:
# Read Parquet metadata only. This is much cheaper than scanning full tables
# and is safe for large canonical KG files.
def parquet_metadata(path: Path) -> dict[str, Any]:
    row: dict[str, Any] = {
        'path': str(path),
        'exists': path.exists(),
        'rows': None,
        'num_columns': None,
        'columns': None,
        'dtypes': None,
        'size_bytes': None,
        'error': None,
    }
    if not path.exists():
        return row
    try:
        row['size_bytes'] = path.stat().st_size
        pf = pq.ParquetFile(path)
        row['rows'] = pf.metadata.num_rows
        schema = pf.schema_arrow
        row['num_columns'] = len(schema.names)
        row['columns'] = ', '.join(schema.names)
        row['dtypes'] = ', '.join(f'{field.name}:{field.type}' for field in schema)
    except Exception as exc:  # noqa: BLE001 - surface corrupt/inaccessible files without aborting
        row['error'] = f'{type(exc).__name__}: {exc}'
    return row


def expected_node_path(root: Path, node_type: str) -> Path:
    return root / 'nodes' / f'{node_type}.parquet'


def expected_edge_path(root: Path, relation: str) -> Path:
    return root / 'edges' / f'{relation}.parquet'


def expected_evidence_path(root: Path, relation: str) -> Path:
    return root / 'evidence' / f'{relation}.parquet'

In [ ]:
if KG_ROOT is None:
    node_files = pd.DataFrame(columns=['node_type', 'primary_ontology', 'exists', 'rows', 'num_columns', 'columns', 'error', 'path'])
else:
    node_file_rows = []
    for row in node_schema.to_dict('records'):
        meta = parquet_metadata(expected_node_path(KG_ROOT, row['node_type']))
        node_file_rows.append({**row, **meta})
    node_files = pd.DataFrame(node_file_rows)

display(node_files[['node_type', 'primary_ontology', 'exists', 'rows', 'num_columns', 'columns', 'error', 'path']])

In [ ]:
if KG_ROOT is None:
    edge_files = pd.DataFrame(columns=['relation', 'x_type', 'y_type', 'kind', 'direct', 'status', 'exists', 'rows', 'num_columns', 'error', 'path'])
    evidence_files = pd.DataFrame(columns=['relation', 'x_type', 'y_type', 'exists', 'rows', 'num_columns', 'error', 'path'])
else:
    edge_file_rows = []
    evidence_file_rows = []
    for row in relation_schema.to_dict('records'):
        edge_file_rows.append({**row, **parquet_metadata(expected_edge_path(KG_ROOT, row['relation']))})
        evidence_file_rows.append({**row, **parquet_metadata(expected_evidence_path(KG_ROOT, row['relation']))})
    edge_files = pd.DataFrame(edge_file_rows)
    evidence_files = pd.DataFrame(evidence_file_rows)

print('Edges')
display(edge_files[['relation', 'x_type', 'y_type', 'kind', 'direct', 'status', 'exists', 'rows', 'num_columns', 'error', 'path']])
print('Evidence')
display(evidence_files[['relation', 'x_type', 'y_type', 'exists', 'rows', 'num_columns', 'error', 'path']])

## Missing / extra files

This section compares the **schema expectation** with the **files visible on disk/mount**.

- Missing expected node files usually indicate a real build/promotion gap.
- Missing relation/evidence files can be normal if a relation is still proposed, deferred, replaced, or not promoted yet.
- Extra files are not automatically wrong; they may be archived, experimental, or newly promoted files not yet reflected in a particular schema summary.

Use this as a navigation aid, not as the sole promotion verdict.


In [ ]:
def parquet_stems(directory: Path) -> set[str]:
    if not directory.exists():
        return set()
    return {p.stem for p in directory.glob('*.parquet') if p.is_file()}

expected_nodes = set(node_schema['node_type'])
expected_relations = set(relation_schema['relation'])
if KG_ROOT is None:
    file_delta = pd.DataFrame([{'kind': 'kg_root', 'status': 'missing', 'name': 'no existing root selected', 'path': None}])
else:
    actual_nodes = parquet_stems(KG_ROOT / 'nodes')
    actual_edges = parquet_stems(KG_ROOT / 'edges')
    actual_evidence = parquet_stems(KG_ROOT / 'evidence')
    rows = []
    for name in sorted(expected_nodes - actual_nodes):
        rows.append({'kind': 'node', 'status': 'missing_expected', 'name': name, 'path': str(KG_ROOT / 'nodes' / f'{name}.parquet')})
    for name in sorted(actual_nodes - expected_nodes):
        rows.append({'kind': 'node', 'status': 'extra_unexpected', 'name': name, 'path': str(KG_ROOT / 'nodes' / f'{name}.parquet')})
    for name in sorted(expected_relations - actual_edges):
        rows.append({'kind': 'edge', 'status': 'missing_expected', 'name': name, 'path': str(KG_ROOT / 'edges' / f'{name}.parquet')})
    for name in sorted(actual_edges - expected_relations):
        rows.append({'kind': 'edge', 'status': 'extra_unexpected', 'name': name, 'path': str(KG_ROOT / 'edges' / f'{name}.parquet')})
    for name in sorted(expected_relations - actual_evidence):
        rows.append({'kind': 'evidence', 'status': 'missing_expected', 'name': name, 'path': str(KG_ROOT / 'evidence' / f'{name}.parquet')})
    for name in sorted(actual_evidence - expected_relations):
        rows.append({'kind': 'evidence', 'status': 'extra_unexpected', 'name': name, 'path': str(KG_ROOT / 'evidence' / f'{name}.parquet')})
    file_delta = pd.DataFrame(rows) if rows else pd.DataFrame(columns=['kind', 'status', 'name', 'path'])

display(file_delta)

## Lightweight schema conformance checks

This section checks whether visible `edges/*.parquet` and `evidence/*.parquet` files contain the baseline required columns.

It is intentionally lightweight:

- it does not validate every endpoint anti-join;
- it does not prove every edge has evidence;
- it does not run heavy relation-specific biological checks.

Those deeper checks belong to the tester/reviewer promotion gates. Here we just surface obvious file-contract mismatches quickly.


In [ ]:
EDGE_REQUIRED = [name for name, _description in EDGE_PARQUET_COLUMNS]
EVIDENCE_REQUIRED = [name for name, _description in EVIDENCE_PARQUET_COLUMNS]


def missing_columns(meta_df: pd.DataFrame, required_columns: list[str]) -> pd.DataFrame:
    rows = []
    for row in meta_df.to_dict('records'):
        if not row.get('exists') or row.get('error'):
            continue
        present = set(str(row.get('columns') or '').split(', ')) if row.get('columns') else set()
        rows.append({
            'name': row.get('relation') or row.get('node_type'),
            'missing_columns': ', '.join(col for col in required_columns if col not in present),
            'extra_columns': ', '.join(col for col in sorted(present - set(required_columns))),
        })
    return pd.DataFrame(rows)

print('Edge file column deltas')
display(missing_columns(edge_files, EDGE_REQUIRED))
print('Evidence file column deltas')
display(missing_columns(evidence_files, EVIDENCE_REQUIRED))

## Optional bounded edge/evidence support check

This is off by default because it can become expensive on the full KG.

Enable only when you explicitly want a bounded diagnostic:

```bash
export JOUVENCE_NOTEBOOK_EVIDENCE_AUDIT=1
```

The full canonical promotion gates still live in the Kanban tester/reviewer tasks; this notebook is for exploration and presentation.


In [ ]:
RUN_EVIDENCE_AUDIT = os.environ.get('JOUVENCE_NOTEBOOK_EVIDENCE_AUDIT', os.environ.get('TXGNN_NOTEBOOK_EVIDENCE_AUDIT')) == '1'
AUDIT_RELATIONS = [r.strip() for r in os.environ.get('JOUVENCE_NOTEBOOK_AUDIT_RELATIONS', os.environ.get('TXGNN_NOTEBOOK_AUDIT_RELATIONS', '')).split(',') if r.strip()]
if not RUN_EVIDENCE_AUDIT:
    print('Evidence support audit skipped. Set JOUVENCE_NOTEBOOK_EVIDENCE_AUDIT=1 to run.')
elif KG_ROOT is None:
    print('Evidence support audit skipped: no existing KG root selected.')
else:
    from manage_db.audit_edge_evidence import audit_edge_evidence

    audit = audit_edge_evidence(KG_ROOT, relations=AUDIT_RELATIONS or None)
    audit_rows = []
    for relation, report in audit.relation_reports.items():
        audit_rows.append({
            'relation': relation,
            'edge_rows': report.edge_rows,
            'evidence_rows': report.evidence_rows,
            'edges_without_evidence': report.edges_without_evidence,
            'evidence_without_edge': report.evidence_without_edge,
            'ok': report.ok,
        })
    display(pd.DataFrame(audit_rows))

## Quick filters for interactive use

These summary tables are the fastest way to explore:

- total schema node/relation counts;
- which node files are visible;
- which relation/evidence files are visible;
- which files have obvious metadata/schema errors.

Suggested workflow:

1. Start with `summary`.
2. Inspect `missing_files` to understand gaps/deferred work.
3. Use `relation_schema` to find the exact endpoint semantics of a relation.
4. Use `edge_files` / `evidence_files` to check whether a relation is actually present in canonical Parquet form.


In [ ]:
summary = pd.DataFrame([
    {'surface': 'node schema entries', 'count': len(node_schema)},
    {'surface': 'relation schema entries', 'count': len(relation_schema)},
    {'surface': 'expected edge columns', 'count': len(EDGE_REQUIRED)},
    {'surface': 'expected evidence columns', 'count': len(EVIDENCE_REQUIRED)},
    {'surface': 'existing KG root selected', 'count': int(KG_ROOT is not None)},
])
display(summary)

## What this notebook shows now, and what is still missing

### What is possible now

- The macOS `gcsfuse` mount is working via `~/mnt/gcs/jouvencekb-kg/v2`, so the notebook can inspect the canonical KG directly instead of relying only on partial local caches.
- The KG schema contract is readable from project code: node types, relation semantics, edge/evidence file expectations, and Parquet column contracts.
- The notebook can quickly distinguish three layers that used to get mixed together:
  1. **Lamin registry/config layer** — useful, but currently missing `lnschema_txgnn` activation.
  2. **Project KG schema layer** — active source of truth in `manage_db.kg_schema` / `manage_db.kg_evidence`.
  3. **Canonical Parquet artifact layer** — actual files under `kg/v2/{nodes,edges,evidence,features}`.
- It is now safe to use N11 as a read-only status/exploration presentation for the KG structure.

### Why `lnschema_txgnn` is not configured

`uv run lamin info` reports the active instance modules as:

```text
bionty, pertdb
```

It does not include `lnschema_txgnn`. Lamin schema modules are instance-level configuration, not just Python files. So even if `manage_db/lnschema_txgnn` exists locally, Lamin raises `ModuleWasntConfigured` until the instance settings include that module. This is a Lamin configuration/activation gap, not evidence that the KG Parquet files are broken.

### What still needs work

- **Lamin layer:** decide whether `lnschema_txgnn` should be enabled in the Lamin instance, and if yes, configure it through the proper Lamin instance settings/admin path.
- **ReMap:** `tf_binds_enhancer` remains the long-running staged/reviewed item. Downstream source-native validation/cleanup stays gated on the active ReMap continuation.
- **Deferred relations/features:** some schema relations or feature candidates may be intentionally absent because their source policy, provenance, or endpoint semantics still need review.
- **Promotion evidence:** this notebook is not a substitute for tester/reviewer gates. It helps you inspect the state, but canonical writes still require validation reports, byte/readback checks, and review handoffs.

### Practical next uses

- Use N11 to explore the KG schema and file presence.
- Use N12 for node feature-layer status (`protein_sequence`, `transcript_sequence`, textual summaries, molecule fingerprints, deferred gene sequence semantics).
- Use the non-ReMap Part 2 notebook for what was actually promoted to canonical edge/evidence form versus what remains deferred.
